# 08 Final Inference

Final inference notebook for the SFT checkpoint produced by `07-sft-exp.ipynb`. It loads `manuka-sft-model` and generates chat-style replies with the same `BOS + prompt + SEP + response` format used during SFT.


In [ ]:
# Run once per fresh Colab runtime if sentencepiece is missing.
try:
    import sentencepiece  # noqa: F401
except ImportError:
    %pip -q install sentencepiece


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/text_dataset/manuka/04-preprocess-exp')
MODEL_DIR = DATA_ROOT / 'manuka-sft-model'
if not MODEL_DIR.exists():
    raise FileNotFoundError(f'{MODEL_DIR} does not exist. Run 07-sft-exp.ipynb first.')
print('MODEL_DIR:', MODEL_DIR)


In [ ]:
import json
import re
import torch
import sentencepiece as spm

SAVE_DIR = MODEL_DIR
device = 'cuda' if torch.cuda.is_available() else 'cpu'

config_path = SAVE_DIR / 'config.json'
model_path = SAVE_DIR / 'model.pt'
if not config_path.exists():
    raise FileNotFoundError(f'{config_path} not found. Train the SFT model first.')
if not model_path.exists():
    raise FileNotFoundError(f'{model_path} not found. Train the SFT model first.')

with open(config_path, encoding='utf-8') as f:
    cfg = json.load(f)
MAX_LEN = int(cfg['max_len'])

tokenizer_name = cfg.get('tokenizer_model', 'spm32k.model')
sp = spm.SentencePieceProcessor(model_file=str(SAVE_DIR / tokenizer_name))
PAD = sp.pad_id(); BOS = sp.bos_id(); EOS = sp.eos_id(); UNK = sp.unk_id()
SEP = sp.piece_to_id('<sep>')
VOCAB_SIZE = sp.get_piece_size()
SPECIALS = {'PAD': PAD, 'BOS': BOS, 'SEP': SEP, 'EOS': EOS, 'UNK': UNK}
SPACE_RGX = re.compile(r'\s+')

if VOCAB_SIZE != int(cfg['vocab_size']):
    raise ValueError(f"Tokenizer vocab size {VOCAB_SIZE} does not match config vocab_size {cfg['vocab_size']}")
if SEP is None or SEP < 0:
    raise ValueError('SFT inference requires a valid <sep> token.')


def clean_text(s: str) -> str:
    s = (s or '').replace('\u200b', ' ').replace('\ufeff', ' ').strip()
    return SPACE_RGX.sub(' ', s)


def encode_text(s: str):
    return sp.encode(clean_text(s), out_type=int)


def decode_text(ids):
    drop = {PAD, BOS, SEP, EOS, UNK}
    return sp.decode([int(i) for i in ids if int(i) not in drop])

print('device:', device)
print('tokenizer:', tokenizer_name)
print('tokenizer vocab size:', VOCAB_SIZE, SPECIALS)
print('MAX_LEN:', MAX_LEN)
print({k: cfg[k] for k in ['vocab_size', 'd_model', 'n_layers', 'n_heads', 'n_kv_heads', 'max_len'] if k in cfg})
print('training phase:', cfg.get('training', {}).get('phase', '<unknown>'))


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint


def rotate_half(x):
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)


def apply_rope(q, k, *, rope_freqs, start_pos=0):
    if q.shape[-1] != k.shape[-1]:
        raise ValueError("q and k must have the same head dimension")

    original_dim = q.shape[-1]
    head_dim = original_dim + original_dim % 2
    if original_dim % 2 != 0:
        q = F.pad(q, (0, 1))
        k = F.pad(k, (0, 1))

    seq_len = q.shape[-2]
    freqs = rope_freqs[start_pos:start_pos + seq_len, :head_dim].to(device=q.device, dtype=torch.float32)
    cos = freqs.cos().to(dtype=q.dtype).view(1, 1, seq_len, head_dim)
    sin = freqs.sin().to(dtype=q.dtype).view(1, 1, seq_len, head_dim)

    q_out = (q * cos) + (rotate_half(q) * sin)
    k_out = (k * cos) + (rotate_half(k) * sin)
    if original_dim % 2 != 0:
        q_out = q_out[..., :original_dim]
        k_out = k_out[..., :original_dim]
    return q_out.contiguous(), k_out.contiguous()


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        normed = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return normed.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)


class CausalSelfAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        dropout,
        max_len,
        n_kv_heads=None,
        qk_norm=True,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
    ):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.n_kv_heads = n_heads if n_kv_heads is None else n_kv_heads
        if n_heads % self.n_kv_heads != 0:
            raise ValueError("n_heads must be divisible by n_kv_heads")
        if rope_scale <= 0:
            raise ValueError("rope_scale must be positive")
        if rope_scaling_type not in {"linear", "ntk"}:
            raise ValueError("rope_scaling_type must be 'linear' or 'ntk'")

        self.head_dim = d_model // n_heads
        self.kv_repeat = n_heads // self.n_kv_heads
        self.rope_theta = rope_theta
        self.rope_scale = rope_scale
        self.rope_scaling_type = rope_scaling_type

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.head_dim) if qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.head_dim) if qk_norm else nn.Identity()
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        self.register_buffer("rope_freqs", self._build_rope_freqs(max_len), persistent=False)

    def _build_rope_freqs(self, max_len):
        head_dim = self.head_dim + self.head_dim % 2
        fractions = torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        inv_freq = 1.0 / (self.rope_theta ** fractions)
        if self.rope_scaling_type == "ntk":
            inv_freq = inv_freq / (self.rope_scale ** fractions)
            positions = torch.arange(max_len, dtype=torch.float32)
        else:
            positions = torch.arange(max_len, dtype=torch.float32) / self.rope_scale
        freqs = torch.einsum("i,j->ij", positions, inv_freq)
        return torch.repeat_interleave(freqs, 2, dim=1)

    def _repeat_kv(self, x):
        if self.kv_repeat == 1:
            return x
        return x.repeat_interleave(self.kv_repeat, dim=1)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        past_len = past_kv[0].shape[-2] if past_kv is not None else 0
        start_pos = past_len
        q, k = apply_rope(q, k, rope_freqs=self.rope_freqs, start_pos=start_pos)

        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)
        present = (k, v) if use_cache else None

        k_full = self._repeat_kv(k)
        v_full = self._repeat_kv(v)
        key_len = k_full.shape[-2]

        causal_needed = T > 1 or past_kv is None
        attn_bias = None
        if attn_mask is not None:
            if attn_mask.dim() != 2:
                raise ValueError("attn_mask must have shape (batch, key_length)")
            if attn_mask.shape[-1] != key_len:
                attn_mask = attn_mask[:, -key_len:]
            pad_mask = (attn_mask == 0).unsqueeze(1).unsqueeze(2)
            attn_bias = torch.zeros((B, 1, T, key_len), device=x.device, dtype=q.dtype)
            attn_bias = attn_bias.masked_fill(pad_mask, torch.finfo(q.dtype).min)
            if causal_needed:
                q_pos = torch.arange(start_pos, start_pos + T, device=x.device).view(T, 1)
                k_pos = torch.arange(key_len, device=x.device).view(1, key_len)
                causal_mask = k_pos > q_pos
                attn_bias = attn_bias.masked_fill(causal_mask.view(1, 1, T, key_len), torch.finfo(q.dtype).min)
                causal_needed = False

        y = F.scaled_dot_product_attention(
            q,
            k_full,
            v_full,
            attn_mask=attn_bias,
            dropout_p=self.attn_drop.p if self.training else 0.0,
            is_causal=causal_needed,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.out_proj(y))
        return y, present


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        mlp_ratio=8/3,
        dropout=0.1,
        max_len=MAX_LEN,
        n_kv_heads=None,
        qk_norm=True,
        residual_scale=1.0,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
    ):
        super().__init__()
        hidden_dim = int(mlp_ratio * d_model)
        hidden_dim = 256 * math.ceil(hidden_dim / 256)
        self.ln1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model,
            n_heads,
            dropout,
            max_len,
            n_kv_heads=n_kv_heads,
            qk_norm=qk_norm,
            rope_theta=rope_theta,
            rope_scale=rope_scale,
            rope_scaling_type=rope_scaling_type,
        )
        self.ln2 = RMSNorm(d_model)
        self.mlp = SwiGLU(d_model, hidden_dim, dropout)
        self.attn_res_scale = nn.Parameter(torch.tensor(float(residual_scale)))
        self.mlp_res_scale = nn.Parameter(torch.tensor(float(residual_scale)))

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        attn_out, present = self.attn(self.ln1(x), attn_mask=attn_mask, past_kv=past_kv, use_cache=use_cache)
        x = x + self.attn_res_scale * attn_out
        x = x + self.mlp_res_scale * self.mlp(self.ln2(x))
        return x, present


class GPTScratch(nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        d_model=640,
        n_layers=12,
        n_heads=10,
        n_kv_heads=10,
        max_len=MAX_LEN,
        dropout=0.1,
        qk_norm=True,
        residual_scale=1.0,
        rope_theta=10000.0,
        rope_scale=1.0,
        rope_scaling_type="linear",
        gradient_checkpointing=False,
    ):
        super().__init__()
        self.max_len = max_len
        self.n_layers = n_layers
        self.gradient_checkpointing = gradient_checkpointing
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.emb_norm = RMSNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                n_heads,
                mlp_ratio=8/3,
                dropout=dropout,
                max_len=max_len,
                n_kv_heads=n_kv_heads,
                qk_norm=qk_norm,
                residual_scale=residual_scale,
                rope_theta=rope_theta,
                rope_scale=rope_scale,
                rope_scaling_type=rope_scaling_type,
            )
            for _ in range(n_layers)
        ])
        self.ln_f = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init)
        self._scale_residual_projections()
        self.head.weight = self.tok_emb.weight

    def _init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _scale_residual_projections(self):
        scale = 0.02 / math.sqrt(2 * self.n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scale)
            nn.init.normal_(block.mlp.w3.weight, mean=0.0, std=scale)

    def forward(self, x, attention_mask=None, past_kv=None, use_cache=False):
        B, T = x.shape
        if past_kv is None and T > self.max_len:
            x = x[:, -self.max_len:]
            T = x.shape[1]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_len:]

        h = self.drop(self.emb_norm(self.tok_emb(x)))
        presents = [] if use_cache else None
        if past_kv is None:
            past_kv = [None] * len(self.blocks)

        for block, layer_past in zip(self.blocks, past_kv):
            if self.gradient_checkpointing and self.training and not use_cache:
                h, present = checkpoint.checkpoint(block, h, attention_mask, layer_past, False, use_reentrant=False)
            else:
                h, present = block(h, attn_mask=attention_mask, past_kv=layer_past, use_cache=use_cache)
            if use_cache:
                presents.append(present)

        logits = self.head(self.ln_f(h))
        if use_cache:
            return logits, presents
        return logits


In [ ]:
model = GPTScratch(
    vocab_size=cfg['vocab_size'],
    d_model=cfg['d_model'],
    n_layers=cfg['n_layers'],
    n_heads=cfg['n_heads'],
    n_kv_heads=cfg.get('n_kv_heads', cfg['n_heads']),
    max_len=cfg['max_len'],
    dropout=cfg.get('dropout', 0.0),
    qk_norm=cfg.get('qk_norm', True),
    residual_scale=cfg.get('residual_scale', 1.0),
    rope_theta=cfg.get('rope_theta', 10000.0),
    rope_scale=cfg.get('rope_scale', 1.0),
    rope_scaling_type=cfg.get('rope_scaling_type', 'linear'),
    gradient_checkpointing=False,
).to(device).eval()

state_dict = torch.load(model_path, map_location='cpu')
model.load_state_dict(state_dict, strict=True)
param_count = sum(p.numel() for p in model.parameters())
print(f"loaded: {param_count / 1e6:.1f}M params | vocab={cfg['vocab_size']} d_model={cfg['d_model']} layers={cfg['n_layers']} heads={cfg['n_heads']} kv_heads={cfg.get('n_kv_heads', cfg['n_heads'])} max_len={cfg['max_len']}")


In [ ]:
# Chat-style SFT inference with KV-cache decoding.
import torch

GEN_CFG = cfg.get('generation', {})
HIST_MAX = cfg.get('max_len', MAX_LEN)
DEFAULT_MAX_NEW = GEN_CFG.get('max_new_tokens', 128)


def banned_tokens_for_no_repeat_ngram(generated, n):
    if n <= 0 or len(generated) < n - 1:
        return set()
    prefix = tuple(generated[-(n - 1):]) if n > 1 else tuple()
    banned = set()
    for i in range(0, len(generated) - n + 1):
        if tuple(generated[i:i + n - 1]) == prefix:
            banned.add(generated[i + n - 1])
    return banned


def apply_repetition_penalty(logits, generated, penalty=1.0):
    if penalty is None or penalty <= 1.0:
        return logits
    for token_id in set(generated):
        if 0 <= token_id < logits.numel():
            logits[token_id] = logits[token_id] * penalty if logits[token_id] < 0 else logits[token_id] / penalty
    return logits


@torch.no_grad()
def sample_next_token(
    logits_row,
    generated,
    *,
    top_p=0.88,
    top_k=60,
    temperature=0.72,
    repetition_penalty=1.12,
    no_repeat_ngram_size=4,
    min_new_tokens=4,
    **_,
):
    logits = logits_row.float().clone()
    for token_id in [PAD, BOS, SEP, UNK]:
        if token_id is not None and token_id >= 0:
            logits[token_id] = -float('inf')
    if len(generated) < min_new_tokens and EOS is not None and EOS >= 0:
        logits[EOS] = -float('inf')
    for token_id in banned_tokens_for_no_repeat_ngram(generated, no_repeat_ngram_size):
        logits[token_id] = -float('inf')
    logits = apply_repetition_penalty(logits, generated, repetition_penalty)
    logits = logits / max(1e-6, float(temperature))

    if top_k and top_k > 0 and top_k < logits.numel():
        kth = torch.topk(logits, top_k).values[-1]
        logits[logits < kth] = -float('inf')

    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    remove = cumulative > top_p
    remove[..., 0] = False
    sorted_probs = sorted_probs.masked_fill(remove, 0)
    if sorted_probs.sum() <= 0 or torch.isnan(sorted_probs.sum()):
        return int(torch.argmax(probs).item())
    sorted_probs = sorted_probs / sorted_probs.sum()
    return int(sorted_idx[torch.multinomial(sorted_probs, 1)].item())


@torch.no_grad()
def generate_reply(prompt: str, max_ctx: int = HIST_MAX, max_new_tokens: int = DEFAULT_MAX_NEW, **sampling_kwargs):
    max_ctx = min(int(max_ctx), MAX_LEN)
    max_new_tokens = min(int(max_new_tokens), max(1, max_ctx - 2))
    prompt = clean_text(prompt)
    prompt_body_budget = max(1, max_ctx - max_new_tokens - 2)
    prompt_body = encode_text(prompt)[-prompt_body_budget:]
    prompt_ids = [BOS] + prompt_body + [SEP]

    x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
    attn = torch.ones_like(x)
    logits, past_kv = model(x, attention_mask=attn, use_cache=True)
    generated = []

    for _ in range(max_new_tokens):
        nxt = sample_next_token(logits[0, -1], generated, **{**GEN_CFG, **sampling_kwargs})
        generated.append(nxt)
        if nxt == EOS:
            break
        x = torch.tensor([[nxt]], dtype=torch.long, device=device)
        logits, past_kv = model(x, past_kv=past_kv, use_cache=True)

    if EOS in generated:
        generated = generated[:generated.index(EOS)]
    return decode_text(generated).strip()


In [ ]:
prompt = '안녕, 오늘 하루는 어땠어?'
print('user:', prompt)
print('model:', generate_reply(prompt, max_new_tokens=128, temperature=0.72, top_p=0.88, top_k=60))


In [ ]:
while True:
    prompt = input('\nprompt: ').strip()
    if prompt.lower() in ['q', 'quit', 'exit']:
        print('bye')
        break
    if not prompt:
        continue
    reply = generate_reply(prompt, max_new_tokens=128, temperature=0.72, top_p=0.88, top_k=60)
    print('model:', reply)
